# Smart Waste Sorting using Computer Vision
### تصنيف المخلفات الذكي باستخدام الرؤية الحاسوبية

**Goal:** Build an image-classification model that looks at a photo of an item of waste
and predicts its category (e.g. `cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`),
so that it can be sorted correctly for recycling.

## خلفية عن المخلفات وطرق التدوير (Recycling Background)

أنواع المخلفات الشائعة وطرق التعامل معها بعد التصنيف:

| الفئة (Class) | الوصف | طريقة المعالجة/التدوير |
|---|---|---|
| **Cardboard / كرتون** | صناديق، ورق مقوى | يُكبس ويُرسل لمصانع الورق لإعادة تصنيعه لعبوات جديدة |
| **Glass / زجاج** | زجاجات، برطمانات | يُغسل ويُكسر (cullet) ثم يُصهر لصناعة زجاج جديد — قابل لإعادة التدوير بلا نهاية |
| **Metal / معدن** | علب الصودا، عبوات الكونسروة | يُفرز (حديدي/غير حديدي بالمغناطيس) ثم يُصهر لإنتاج سبائك جديدة |
| **Paper / ورق** | جرائد، مجلات | يُنقع ويُحول للب الورق (pulping) لإنتاج ورق معاد التدوير |
| **Plastic / بلاستيك** | زجاجات، أكياس | يُفرز حسب نوع البلاستيك (PET, HDPE ..) ثم يُغسل ويُجرش (shredding) ويُعاد تصنيعه كحبيبات (pellets) |
| **Trash / مخلفات عامة (غير قابلة للتدوير)** | مخلفات مختلطة/ملوثة | تُرسل لمكب النفايات الصحي أو للحرق لتوليد الطاقة (waste-to-energy) |

الفكرة العامة لخط المعالجة (Materials Recovery Facility pipeline):
1. **الفرز الأولي (Sorting):** يدويًا أو بالرؤية الحاسوبية (هذا المشروع) لتحديد نوع المادة.
2. **الفصل الآلي:** مغناطيس للمعادن الحديدية، تيار دوامي (eddy current) للألمنيوم، فرز بصري للبلاستيك والزجاج.
3. **التنظيف/الغسيل:** إزالة الملوثات والبقايا العضوية.
4. **التجهيز:** تقطيع/كبس/جرش حسب نوع المادة.
5. **إعادة التصنيع:** تحويل المادة لمنتج وسيط (حبيبات بلاستيك، سبائك معدن، لب ورق، زجاج مكسور) يُباع لمصانع التصنيع.

This notebook automates **step 1** using a CNN (transfer learning) so a camera/phone photo can be
routed to the correct recycling stream automatically.

---
## Dataset
We use the well-known **TrashNet** dataset (Gary Thung & Mindy Yang, 2016) — ~2527 images across
6 classes: `cardboard, glass, metal, paper, plastic, trash`.

- Paper/repo: https://github.com/garythung/trashnet
- Mirrors also exist on Kaggle (e.g. "Garbage Classification" dataset) if you prefer to use
  `kagglehub`/`opendatasets`.

> **Note:** Because this notebook may run in an offline/restricted environment, the data-loading
> cell below tries a few strategies (git clone, kagglehub, or a local folder you already downloaded)
> and clearly tells you which one to use. Simply point `DATA_DIR` at a folder with one subfolder
> per class (standard `ImageFolder` layout) and everything else runs unchanged.


## 1. Importing modules

In [ ]:

import os
import json
import shutil
import pathlib
import random
import subprocess
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
)
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))


## 2. Loading data

`DATA_DIR` must contain one sub-folder per class, e.g.:

```
data/
  cardboard/
  glass/
  metal/
  paper/
  plastic/
  trash/
```

Pick ONE of the options below.


In [ ]:

DATA_DIR = "data/dataset-resized"  # <-- change if your folder is named differently

# ---- Option A: clone TrashNet from GitHub (small metadata only; large image zip
#      is stored via git-lfs / external link on that repo, so this may only fetch
#      a placeholder in restricted environments) ----
if not os.path.isdir(DATA_DIR):
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/garythung/trashnet.git", "data/trashnet_repo"],
            check=True
        )
        candidate = "data/trashnet_repo/data/dataset-resized"
        if os.path.isdir(candidate):
            DATA_DIR = candidate
    except Exception as e:
        print("Git clone option failed / not available here:", e)

# ---- Option B: Kaggle download (uncomment if you have a kaggle.json configured) ----
# import kagglehub
# path = kagglehub.dataset_download("asdasdasasdas/garbage-classification")
# DATA_DIR = path

# ---- Option C: manual download ----
# 1) Download from https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification
#    or https://github.com/garythung/trashnet (see their README for the dataset-resized.zip link)
# 2) Unzip into ./data/dataset-resized so it matches the class-per-folder layout above.

assert os.path.isdir(DATA_DIR), (
    f"DATA_DIR='{DATA_DIR}' not found. Please download TrashNet (see markdown above) "
    "and update DATA_DIR to point at the extracted folder."
)

CLASS_NAMES = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
print("Classes found:", CLASS_NAMES)


## 3. Analysis and Visualization (EDA)

In [ ]:

# Build a dataframe of filepaths + labels
records = []
for cls in CLASS_NAMES:
    cls_dir = os.path.join(DATA_DIR, cls)
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            records.append({"filepath": os.path.join(cls_dir, fname), "label": cls})

df = pd.DataFrame(records)
print("Total images:", len(df))
df.head()


In [ ]:

# Class distribution
plt.figure(figsize=(8, 4))
df["label"].value_counts().plot(kind="bar", color="seagreen")
plt.title("Class distribution / توزيع الفئات")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.tight_layout()
plt.show()


In [ ]:

# Sample grid of images per class
fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(12, 3 * len(CLASS_NAMES)))
for i, cls in enumerate(CLASS_NAMES):
    sample_paths = df[df.label == cls]["filepath"].sample(4, random_state=SEED).tolist()
    for j, p in enumerate(sample_paths):
        img = keras.utils.load_img(p)
        axes[i, j].imshow(img)
        axes[i, j].axis("off")
        if j == 0:
            axes[i, j].set_ylabel(cls)
    axes[i, 0].set_title(cls, loc="left", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Preprocessing

In [ ]:

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED
)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

label_to_index = {c: i for i, c in enumerate(CLASS_NAMES)}
index_to_label = {i: c for c, i in label_to_index.items()}

with open("class_indices.json", "w") as f:
    json.dump(index_to_label, f, ensure_ascii=False, indent=2)


In [ ]:

def make_dataset(dataframe, training=False):
    paths = dataframe["filepath"].values
    labels = dataframe["label"].map(label_to_index).values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        return img, label

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        augment = keras.Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
            layers.RandomContrast(0.1),
        ])
        ds = ds.map(lambda x, y: (augment(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(1024, seed=SEED)

    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)


## 5. Model building

**Baseline:** a small CNN trained from scratch.
**Final model:** transfer learning on `MobileNetV2` (ImageNet weights), fine-tuned — this
generally reaches much higher accuracy with far less data/epochs, which fits a 6-class,
~2500-image dataset like TrashNet well.


In [ ]:

# ---- Baseline CNN ----
def build_baseline(num_classes):
    model = keras.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        layers.Rescaling(1.0 / 127.5, offset=-1),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

baseline_model = build_baseline(len(CLASS_NAMES))
baseline_history = baseline_model.fit(train_ds, validation_data=val_ds, epochs=5)


In [ ]:

# ---- Final model: MobileNetV2 transfer learning ----
def build_transfer_model(num_classes, fine_tune_at=None):
    base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
    base.trainable = fine_tune_at is not None
    if fine_tune_at is not None:
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False

    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    return model, base

model, base_model = build_transfer_model(len(CLASS_NAMES))
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


In [ ]:

# ---- Hyperparameter tuning (simple grid over learning rate + dropout) ----
# For a 6-class small dataset a light manual search is usually enough; for larger
# search spaces, swap this loop for keras_tuner.
best_val_acc = 0.0
best_lr = None
for lr in [1e-3, 5e-4, 1e-4]:
    m, _ = build_transfer_model(len(CLASS_NAMES))
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    hist = m.fit(train_ds, validation_data=val_ds, epochs=3, verbose=0)
    val_acc = max(hist.history["val_accuracy"])
    print(f"lr={lr:.0e} -> val_accuracy={val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_lr = lr

print("Best learning rate:", best_lr, "val_accuracy:", best_val_acc)


In [ ]:

# ---- Final training run with best hyperparameters (head-only, then fine-tune) ----
model, base_model = build_transfer_model(len(CLASS_NAMES))
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=best_lr or 1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

history_head = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)


In [ ]:

# ---- Fine-tune: unfreeze the top layers of MobileNetV2 ----
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30  # unfreeze last 30 layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)


In [ ]:

# ---- Training curves ----
def plot_history(histories, keys=("accuracy", "val_accuracy")):
    plt.figure(figsize=(8, 4))
    epoch_offset = 0
    for h in histories:
        for k in keys:
            plt.plot(range(epoch_offset, epoch_offset + len(h.history[k])), h.history[k], label=k)
        epoch_offset += len(h.history[keys[0]])
    plt.legend()
    plt.xlabel("Epoch")
    plt.title("Accuracy over training (head-training + fine-tuning)")
    plt.show()

plot_history([history_head, history_fine])


## 6. Evaluation

We report **accuracy** and **macro F1-score** because the classes are imbalanced
(TrashNet has fewer `trash`/`metal` images than `paper`/`glass`), so macro-F1 gives a
fairer picture than accuracy alone. A confusion matrix highlights which materials are
visually confused (e.g. clear plastic vs. glass), which matters a lot for correct sorting.


In [ ]:

y_true = []
y_pred = []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(preds, axis=1).tolist())

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
print("Macro F1:", f1_score(y_true, y_pred, average="macro"))


In [ ]:

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(7, 7))
disp.plot(ax=ax, cmap="Greens", xticks_rotation=45)
plt.title("Confusion matrix — waste category classification")
plt.tight_layout()
plt.show()


## 7. Exporting the model

We save:
1. The trained Keras model (`waste_classifier.keras` — the modern, recommended format).
2. `class_indices.json` mapping output index -> class name (already written in step 4).

Both files are consumed directly by `model/predict.py` in the Streamlit app.


In [ ]:

EXPORT_DIR = "exported_model"
os.makedirs(EXPORT_DIR, exist_ok=True)

model.save(os.path.join(EXPORT_DIR, "waste_classifier.keras"))
shutil.copy("class_indices.json", os.path.join(EXPORT_DIR, "class_indices.json"))

print("Saved to:", os.listdir(EXPORT_DIR))


In [ ]:

# Sanity check: reload and predict on one test image
reloaded = keras.models.load_model(os.path.join(EXPORT_DIR, "waste_classifier.keras"))

sample_path = test_df.iloc[0]["filepath"]
img = keras.utils.load_img(sample_path, target_size=IMG_SIZE)
arr = keras.utils.img_to_array(img)
arr = preprocess_input(arr)
arr = np.expand_dims(arr, axis=0)

pred = reloaded.predict(arr, verbose=0)[0]
pred_idx = int(np.argmax(pred))
print("True label:", test_df.iloc[0]["label"])
print("Predicted:", index_to_label[pred_idx], "confidence:", float(pred[pred_idx]))


---
## Next step
Copy `exported_model/waste_classifier.keras` and `exported_model/class_indices.json`
into the Streamlit project's `model/` folder (see the accompanying GitHub codebase),
then run:

```bash
streamlit run app.py
```
